In [1]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.external_data_encyclopedia import ExternalDataEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

ed = ExternalDataEncyclopedia(
    external_data_path=Path(r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data")
)

In [2]:
exports_folder = Path("refine_plus_export_pool")

In [4]:
from pathlib import Path
from typing import List, Dict

import igl
import numpy as np
import polars as pl
from polars import Series
from polars.dataframe.frame import DataFrame

from boulder_statistics.refinement_plus.bulk_parse_data_vnir_maps import VNIR_MEASUREMENT_NAMES, VNIR_SIGMA_MEASUREMENT_NAMES
from boulder_statistics.refinement_plus.facet_parser import FacetParser
from boulder_statistics.refinement_plus.bulk_parse_data_vnir_maps import (
    FACET_SHAPE_MODELS,
    DataVnirMaps,
)
from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from boulder_statistics.refinement_plus.refinement_chunking import ChunkingTools


mesh_folder = Path(
    r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data\bennu_models"
)

measurement_types: list[str] = VNIR_MEASUREMENT_NAMES + VNIR_SIGMA_MEASUREMENT_NAMES

data_vnir_maps: DataFrame = DataVnirMaps.bulk_parse(ed)
mission_phase: str = "detailed_survey"

vnir_spectral_export_path: Path = exports_folder / f"02_add_vnir_maps_{mission_phase}"

print(f"Running for mission phase {mission_phase}")
mesh_file_path: Path = mesh_folder / FACET_SHAPE_MODELS[mission_phase]

# Load original mesh arrays for libigl
V, F = igl.read_triangle_mesh(mesh_file_path)

# Load processed mesh tables
points, tris = FacetParser.load_mesh(mesh_file_path)

facets: DataFrame = data_vnir_maps.filter(
    pl.col("mission_phase") == mission_phase
)

facets

Running for mission phase detailed_survey


facet_num,mission_phase,band depth,reflectance,slope1 poly,slope2 poly,sigma band depth,sigma reflectance,sigma slope1 poly,sigma slope2 poly,x,y,z,count
i32,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
296,"""detailed_survey""",0.132034,0.039145,0.000183,0.003867,0.011621,0.000171,0.000078,0.00003,-0.135737,0.109813,0.142843,6
43157,"""detailed_survey""",0.125791,0.037972,0.004424,-0.000433,0.013097,0.000202,0.000089,0.000034,-0.106583,-0.2396,-0.038707,6
24258,"""detailed_survey""",0.153838,0.039345,-0.013466,-0.01977,0.01421,0.000172,0.000078,0.000029,0.07525,-0.113247,0.19566,6
12864,"""detailed_survey""",0.149957,0.039142,-0.025307,-0.020038,0.011165,0.00014,0.000063,0.000024,-0.027787,0.093113,0.220253,6
6691,"""detailed_survey""",0.128517,0.037901,0.009974,0.005647,0.010303,0.000156,0.00007,0.000027,-0.081927,0.133507,0.17858,6
…,…,…,…,…,…,…,…,…,…,…,…,…,…
13362,"""detailed_survey""",0.150756,0.03955,-0.014893,-0.012122,0.009515,0.000122,0.000055,0.000021,-0.01897,0.111387,0.209783,6
2842,"""detailed_survey""",0.105424,0.036272,0.007769,0.022282,0.008983,0.000139,0.000066,0.000026,-0.109227,0.132553,0.154267,6
20551,"""detailed_survey""",0.145236,0.037517,-0.011344,-0.005502,0.009949,0.000126,0.000057,0.000022,0.05718,0.064383,0.21349,6


In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Scatter plot
ax.scatter(facets["x"], facets["y"], facets["z"], color='blue', marker='o')

# Labels
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

plt.show()

In [3]:
from pathlib import Path
from typing import List, Dict

import igl
import numpy as np
import polars as pl
from polars import Series
from polars.dataframe.frame import DataFrame

from boulder_statistics.refinement_plus.bulk_parse_data_vnir_maps import VNIR_MEASUREMENT_NAMES, VNIR_SIGMA_MEASUREMENT_NAMES
from boulder_statistics.refinement_plus.facet_parser import FacetParser
from boulder_statistics.refinement_plus.bulk_parse_data_vnir_maps import (
    FACET_SHAPE_MODELS,
    DataVnirMaps,
)
from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from boulder_statistics.refinement_plus.refinement_chunking import ChunkingTools


mesh_folder = Path(
    r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data\bennu_models"
)

measurement_types: list[str] = VNIR_MEASUREMENT_NAMES + VNIR_SIGMA_MEASUREMENT_NAMES

data_vnir_maps: DataFrame = DataVnirMaps.bulk_parse(ed)
mission_phases: Series = data_vnir_maps["mission_phase"].unique()


for mission_phase in mission_phases:
    vnir_spectral_export_path: Path = exports_folder / f"02_add_vnir_maps_{mission_phase}"
    if vnir_spectral_export_path.exists():
        continue

    print(f"Running for mission phase {mission_phase}")
    mesh_file_path: Path = mesh_folder / FACET_SHAPE_MODELS[mission_phase]

    # Load original mesh arrays for libigl
    V, F = igl.read_triangle_mesh(mesh_file_path)

    # Load processed mesh tables
    points, tris = FacetParser.load_mesh(mesh_file_path)

    facets: DataFrame = data_vnir_maps.filter(
        pl.col("mission_phase") == mission_phase
    )

    # ------------------------------------------------------------
    # Associate facets with mesh triangles using point-to-mesh distance
    # ------------------------------------------------------------

    P = (
        facets
        .select(["x", "y", "z"])
        .to_numpy()
        .astype(np.float64)
    )

    sqrD, tri_idx, closest_points = igl.point_mesh_squared_distance(
        P,
        V,
        F
    )

    associate_distance = np.sqrt(sqrD)

    facet_nums = (
        facets
        .get_column("facet_num")
        .to_numpy()
        .astype(np.int32)
    )

    facet_nums_to_tri_nums_lookup = pl.DataFrame(
        {
            "facet_num": facet_nums,
            "tri_num": tri_idx.astype(np.int32),
            "associate_distance": associate_distance,
        }
    )

    # Sanity checks
    if not np.all(
        facet_nums_to_tri_nums_lookup["associate_distance"].to_numpy()
        < 1e-5
    ):
        print("Facets not lining up, skipping")
        continue

    duplicates = (
        facet_nums_to_tri_nums_lookup
        .group_by("tri_num")
        .len()
        .filter(pl.col("len") > 1)
    )

    if duplicates.height != 0:
        print("duplicate triangles found!, skipping")
        print(duplicates)

        continue


    # ------------------------------------------------------------
    # Join facet measurements onto triangles
    # ------------------------------------------------------------

    tris_facets_joined: DataFrame = (
        tris
        .join(
            facet_nums_to_tri_nums_lookup,
            on="tri_num"
        )
        .join(
            facets,
            on="facet_num"
        )
    )

    if tris_facets_joined["tri_num"].n_unique() != tris_facets_joined.height:
        print("Non unique tris detected, skipping")
        continue


    # ------------------------------------------------------------
    # Create triangle-indexed measurement arrays
    # ------------------------------------------------------------

    measurement_type_triangle_values_dict: Dict[str, np.ndarray] = {}

    for measurement_type in measurement_types:

        triangle_values = np.full(
            tris.height,
            np.nan,
            dtype=np.float32,
        )

        triangle_values[
            tris_facets_joined["tri_num"].to_numpy()
        ] = (
            tris_facets_joined
            .get_column(measurement_type)
            .to_numpy()
            .astype(np.float32)
        )

        measurement_type_triangle_values_dict[measurement_type] = (
            triangle_values
        )


    # ------------------------------------------------------------
    # Rasterisation
    # ------------------------------------------------------------

    def handle_chunk(chunk: QCubeChunk) -> List[np.ndarray]:

        triangle_index_image: np.ndarray = FacetParser.rasterize_facets(
            points,
            tris,
            chunk,
        )

        measurement_types_results: List[np.ndarray] = []

        mask = ~np.isnan(triangle_index_image)

        for measurement_type in measurement_types:

            output = np.full(
                triangle_index_image.shape,
                np.nan,
                dtype=np.float32,
            )

            output[mask] = (
                measurement_type_triangle_values_dict[measurement_type]
                [
                    triangle_index_image[mask]
                    .astype(np.int32)
                ]
            )

            measurement_types_results.append(output.T)

        return measurement_types_results


    ChunkingTools.bulk_append_by_chunks(
        dp.combined_atlas.select("i", "j", "face"),
        vnir_spectral_export_path,
        [f"{mission_phase} {type_name}" for type_name in measurement_types],
        handle_chunk,
        chunks=QCubeChunk.generate(depth=2),
    )

c:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\.venv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


Running for mission phase reconb
Facets not lining up, skipping
Running for mission phase detailed_survey


Processing chunks: 100%|██████████| 96/96 [02:12<00:00,  1.38s/it]
